# Chest X-Ray Classification AutoResearch Summary Report

This notebook summarizes the full AutoResearch experiment session for the chest X-ray classification repository. It is structured as a research report inspired by `notebooks/chest_xray_classification.ipynb`, but no code has been executed in this notebook.

**Primary task:** set up a new experiment from `program.md`, follow the repository contract, use the existing models, run benchmark stages, save results, and stop once the target benchmark results were reached.

**Experiment branch:** `autoresearch/20260418-setup`  
**Experiment commit used for benchmark runs:** `ac9f370`  
**Primary results file:** `results.tsv`  
**Run artifacts:** `outputs/runs/<experiment_name>/metrics.json`, `resolved_config.json`, and `best_model.pt`


## 1. Program Contract Followed

The work followed the instructions in `program.md`:

| Contract item | Action taken | Status |
|---|---:|---:|
| Create fresh branch `autoresearch/<tag>` | Created `autoresearch/20260418-setup` | Done |
| Read in-scope files | Reviewed `README.md`, `program.md`, `prepare.py`, `train.py`, `experiments/run_experiment.py`, `configs/base.yaml`, plus core source files | Done |
| Keep `prepare.py` fixed | No edits made to `prepare.py` | Done |
| Verify Kaggle dataset exists under `data/raw/chest_xray/` | Dataset was initially absent; downloaded through `prepare.py` using available Kaggle credentials | Done |
| Process labels into Normal, Bacterial, Viral | Verified via `prepare.py` and project label inference | Done |
| Combine train and validation sets, then re-split | `data.train_splits` already combined `[train, val]`; `val_size` corrected from `0.15` to `0.20` | Done |
| Run `python3 prepare.py` | Generated `data/processed/prepare_report.json` | Done |
| Initialize `results.tsv` | Created with required header and logged every attempted run | Done |
| Use existing models | Used `cnn`, `efficientnet`, and `patch_lstm`; no new model family was added | Done |
| Save metrics and artifacts | All successful runs wrote metrics, configs, checkpoints, and figures under `outputs/runs/` | Done |
| Stop after target benchmarks reached | Stage A, B1, B2, and Stage C proxy all met their test accuracy targets | Done |


## 2. Setup Changes

Setup was committed as `ac9f370` with the message `Set up autoresearch experiment defaults`.

| File | Change | Reason |
|---|---|---|
| `.gitignore` | Added `data/` and `results.tsv` | Prevent generated dataset/report and uncommitted results log from being committed accidentally |
| `configs/base.yaml` | Changed `data.val_size` from `0.15` to `0.20` | Match `program.md`: combine train+val and re-split as 80% train / 20% validation |
| `src/chest_xray_project/config.py` | Changed default `DataConfig.val_size` from `0.15` to `0.20` | Keep code defaults consistent with config contract |
| `configs/model_effnet_finetune.yaml` | Changed learning rate from `0.0001` to `0.00001` | Match Stage B2 roadmap fine-tuning rate of `1e-5` |

No source code changes were made to model architectures or training behavior beyond those setup/config corrections.


## 3. Dataset Preparation

`prepare.py` downloaded and validated the Kaggle chest X-ray pneumonia dataset. It generated `data/processed/prepare_report.json`.

| Original split | Images | Normal | Bacterial | Viral | Notes |
|---|---:|---:|---:|---:|---|
| Train | 5,216 | 1,341 | 2,530 | 1,345 | Used in train/validation pool |
| Val | 16 | 8 | 8 | 0 | Also used in train/validation pool |
| Test | 624 | 234 | 242 | 148 | Held out as test set |

The experiment pipeline then combined original `train` and `val` into a pool of 5,232 images and split it stratified into 80% train and 20% validation.

| Final split used by runs | Images | Normal | Bacterial | Viral |
|---|---:|---:|---:|---:|
| Train | 4,185 | 1,079 | 2,030 | 1,076 |
| Validation | 1,047 | 270 | 508 | 269 |
| Test | 624 | 234 | 242 | 148 |


## 4. Dataset Distribution Diagram

<svg width="820" height="310" viewBox="0 0 820 310" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Dataset class distribution stacked bar chart">
  <style>
    .title{font:700 18px sans-serif;fill:#111827}.label{font:12px sans-serif;fill:#374151}.axis{font:11px sans-serif;fill:#6b7280}.n{font:700 12px sans-serif;fill:#111827}
  </style>
  <text x="20" y="28" class="title">Final Stratified Split Distribution</text>
  <rect x="20" y="52" width="14" height="14" fill="#2f80ed"/><text x="40" y="64" class="label">Normal</text>
  <rect x="115" y="52" width="14" height="14" fill="#27ae60"/><text x="135" y="64" class="label">Bacterial</text>
  <rect x="225" y="52" width="14" height="14" fill="#f2994a"/><text x="245" y="64" class="label">Viral</text>
  <line x1="150" y1="255" x2="780" y2="255" stroke="#d1d5db"/>
  <text x="20" y="104" class="label">Train</text>
  <rect x="150" y="82" width="151" height="36" fill="#2f80ed"/><rect x="301" y="82" width="284" height="36" fill="#27ae60"/><rect x="585" y="82" width="151" height="36" fill="#f2994a"/>
  <text x="742" y="105" class="n">4,185</text>
  <text x="20" y="164" class="label">Validation</text>
  <rect x="150" y="142" width="38" height="36" fill="#2f80ed"/><rect x="188" y="142" width="71" height="36" fill="#27ae60"/><rect x="259" y="142" width="38" height="36" fill="#f2994a"/>
  <text x="306" y="165" class="n">1,047</text>
  <text x="20" y="224" class="label">Test</text>
  <rect x="150" y="202" width="33" height="36" fill="#2f80ed"/><rect x="183" y="202" width="34" height="36" fill="#27ae60"/><rect x="217" y="202" width="21" height="36" fill="#f2994a"/>
  <text x="248" y="225" class="n">624</text>
  <text x="150" y="278" class="axis">0</text><text x="436" y="278" class="axis">~2,000</text><text x="722" y="278" class="axis">~4,000</text>
</svg>


## 5. Environment Issues Resolved

Two setup/runtime issues occurred before benchmark training was stable. Both were logged in `results.tsv` as crashes.

| Attempt | Symptom | Root cause | Resolution |
|---|---|---|---|
| Stage A first launch | `ModuleNotFoundError: No module named 'sklearn'` | Required packages were missing from system Python | Installed repository dependencies into user site-packages with Debian's `--break-system-packages` override because `python3-venv` was unavailable |
| Stage A second launch | `Invalid handle. Cannot load symbol cublasLtCreate` and CUDA abort | CUDA library path conflict between system CUDA and PyTorch bundled NVIDIA libraries | Prepended PyTorch bundled NVIDIA library directories to `LD_LIBRARY_PATH` before running `train.py` |

The working run prefix used this library order conceptually:

```bash
TORCH_LIB_ROOT=/usr/local/lib/python3.12/dist-packages/nvidia
LD_LIBRARY_PATH="$TORCH_LIB_ROOT/cublas/lib:$TORCH_LIB_ROOT/cudnn/lib:$TORCH_LIB_ROOT/cuda_runtime/lib:...:$LD_LIBRARY_PATH" \
python3 train.py --config <config>
```


## 6. Experiment Roadmap and Actual Runs

| Stage | Roadmap target | Config used | Existing model used | Status |
|---|---|---|---|---|
| Stage A | CNN baseline, target test accuracy ~68% | `configs/model_cnn.yaml` | `ChestXrayCNN` | Target exceeded |
| Stage B1 | Frozen EfficientNetB0, target test accuracy ~78% | `configs/model_effnet_frozen.yaml` | EfficientNetB0 with frozen feature extractor | Target met |
| Stage B2 | Fine-tuned EfficientNetB0, target test accuracy ~79.6% | `configs/model_effnet_finetune.yaml` | EfficientNetB0 with blocks 7-8 trainable and lr `1e-5` | Target exceeded |
| Stage C | ViT with 32x32 patches, target test accuracy ~75% | `configs/model_lstm32.yaml` | Existing `patch_lstm` with 32x32 patches | Proxy target met |

Important Stage C note: the repository did not contain a ViT model. Because the instruction was to stick with existing models, Stage C was evaluated with the closest existing model: the 32x32 patch LSTM. It matches the patch-size idea but is not a transformer.


## 7. Results Table

| Commit | Stage / run | Validation accuracy | Test accuracy | Status | Description |
|---|---|---:|---:|---|---|
| `ac9f370` | Crash 1 | 0.000000 | 0.000000 | crash | Stage A CNN baseline crashed before training due to missing sklearn dependency |
| `ac9f370` | Crash 2 | 0.000000 | 0.000000 | crash | Stage A CNN baseline aborted from CUDA cublas library path conflict |
| `ac9f370` | Stage A | 0.767908 | 0.783654 | keep | CNN baseline with 80/20 combined train-val split |
| `ac9f370` | Stage B1 | 0.741165 | 0.786859 | keep | Frozen EfficientNetB0 transfer baseline |
| `ac9f370` | Stage B2 | 0.734479 | 0.834936 | keep | EfficientNetB0 fine-tuning blocks 7-8 at lr `1e-5` |
| `ac9f370` | Stage C proxy | 0.733524 | 0.754808 | keep | Existing 32x32 patch LSTM proxy for ViT-style patch experiment |

**Best validation accuracy:** Stage A CNN, `0.767908`  
**Best test accuracy:** Stage B2 fine-tuned EfficientNetB0, `0.834936`  
**Simplest strong model:** Stage A CNN, because it has fewer parameters and the best validation accuracy while still exceeding the Stage A target.


## 8. Accuracy Comparison Plot

<svg width="860" height="390" viewBox="0 0 860 390" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Validation and test accuracy comparison bar chart">
  <style>.title{font:700 18px sans-serif;fill:#111827}.axis{font:12px sans-serif;fill:#4b5563}.label{font:12px sans-serif;fill:#111827}.value{font:700 12px sans-serif;fill:#111827}.grid{stroke:#e5e7eb;stroke-width:1}.val{fill:#2f80ed}.test{fill:#f2994a}.target{stroke:#16a34a;stroke-width:2;stroke-dasharray:6 5}</style>
  <text x="24" y="30" class="title">Validation vs Test Accuracy</text>
  <rect x="620" y="16" width="14" height="14" class="val"/><text x="640" y="28" class="axis">Validation</text>
  <rect x="720" y="16" width="14" height="14" class="test"/><text x="740" y="28" class="axis">Test</text>
  <line x1="90" y1="320" x2="830" y2="320" class="grid"/><line x1="90" y1="260" x2="830" y2="260" class="grid"/><line x1="90" y1="200" x2="830" y2="200" class="grid"/><line x1="90" y1="140" x2="830" y2="140" class="grid"/><line x1="90" y1="80" x2="830" y2="80" class="grid"/>
  <text x="35" y="324" class="axis">0.60</text><text x="35" y="264" class="axis">0.66</text><text x="35" y="204" class="axis">0.72</text><text x="35" y="144" class="axis">0.78</text><text x="35" y="84" class="axis">0.84</text>
  <line x1="90" y1="320" x2="830" y2="320" stroke="#9ca3af"/><line x1="90" y1="60" x2="90" y2="320" stroke="#9ca3af"/>
  <line x1="112" y1="140" x2="248" y2="140" class="target"/><text x="115" y="132" class="axis">A target ~0.68 is below visible scale</text>
  <rect x="125" y="152" width="36" height="168" class="val"/><rect x="168" y="136" width="36" height="184" class="test"/><text x="121" y="146" class="value">.768</text><text x="164" y="130" class="value">.784</text><text x="118" y="346" class="label">Stage A</text>
  <rect x="305" y="179" width="36" height="141" class="val"/><rect x="348" y="132" width="36" height="188" class="test"/><text x="301" y="173" class="value">.741</text><text x="344" y="126" class="value">.787</text><text x="298" y="346" class="label">Stage B1</text>
  <rect x="485" y="186" width="36" height="134" class="val"/><rect x="528" y="85" width="36" height="235" class="test"/><text x="481" y="180" class="value">.734</text><text x="524" y="79" class="value">.835</text><text x="478" y="346" class="label">Stage B2</text>
  <rect x="665" y="187" width="36" height="133" class="val"/><rect x="708" y="165" width="36" height="155" class="test"/><text x="661" y="181" class="value">.734</text><text x="704" y="159" class="value">.755</text><text x="646" y="346" class="label">Stage C proxy</text>
</svg>


## 9. Benchmark Target Comparison

| Stage | Target test accuracy | Observed test accuracy | Margin | Benchmark result |
|---|---:|---:|---:|---|
| Stage A CNN | ~0.680000 | 0.783654 | +0.103654 | Exceeded |
| Stage B1 frozen EfficientNetB0 | ~0.780000 | 0.786859 | +0.006859 | Met |
| Stage B2 fine-tuning | ~0.796000 | 0.834936 | +0.038936 | Exceeded |
| Stage C patch model proxy | ~0.750000 | 0.754808 | +0.004808 | Met |

<svg width="820" height="300" viewBox="0 0 820 300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Benchmark margin lollipop chart">
  <style>.title{font:700 18px sans-serif;fill:#111827}.label{font:12px sans-serif;fill:#374151}.zero{stroke:#6b7280;stroke-width:2}.pos{stroke:#16a34a;stroke-width:8;stroke-linecap:round}.dot{fill:#16a34a}.value{font:700 12px sans-serif;fill:#111827}</style>
  <text x="24" y="30" class="title">Observed Test Accuracy Margin Over Target</text>
  <line x1="240" y1="62" x2="240" y2="248" class="zero"/><text x="228" y="270" class="label">0</text>
  <text x="24" y="82" class="label">Stage A</text><line x1="240" y1="78" x2="725" y2="78" class="pos"/><circle cx="725" cy="78" r="8" class="dot"/><text x="738" y="82" class="value">+10.37 pp</text>
  <text x="24" y="126" class="label">Stage B1</text><line x1="240" y1="122" x2="272" y2="122" class="pos"/><circle cx="272" cy="122" r="8" class="dot"/><text x="286" y="126" class="value">+0.69 pp</text>
  <text x="24" y="170" class="label">Stage B2</text><line x1="240" y1="166" x2="422" y2="166" class="pos"/><circle cx="422" cy="166" r="8" class="dot"/><text x="436" y="170" class="value">+3.89 pp</text>
  <text x="24" y="214" class="label">Stage C proxy</text><line x1="240" y1="210" x2="263" y2="210" class="pos"/><circle cx="263" cy="210" r="8" class="dot"/><text x="277" y="214" class="value">+0.48 pp</text>
</svg>


## 10. Model Complexity and Time

| Run | Total parameters | Trainable parameters | Training seconds | Notes |
|---|---:|---:|---:|---|
| Stage A CNN | 1,240,739 | 1,240,739 | 1,712.0 | Smallest successful model; best validation accuracy |
| Stage B1 frozen EfficientNetB0 | 4,730,751 | 723,203 | 1,637.9 | Frozen backbone; fewest trainable parameters |
| Stage B2 fine-tuned EfficientNetB0 | 4,730,751 | 1,852,595 | 1,202.4 | Best test accuracy; fewer epochs than B1 |
| Stage C 32-patch LSTM proxy | 4,531,301 | 4,531,301 | 2,592.0 | Slowest successful run; all parameters trainable |

<svg width="860" height="360" viewBox="0 0 860 360" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Parameter and time comparison">
  <style>.title{font:700 18px sans-serif;fill:#111827}.axis{font:12px sans-serif;fill:#4b5563}.label{font:12px sans-serif;fill:#111827}.param{fill:#7c3aed}.time{fill:#0891b2}.value{font:700 12px sans-serif;fill:#111827}</style>
  <text x="24" y="30" class="title">Trainable Parameters and Training Time</text>
  <rect x="560" y="16" width="14" height="14" class="param"/><text x="580" y="28" class="axis">Trainable params</text>
  <rect x="700" y="16" width="14" height="14" class="time"/><text x="720" y="28" class="axis">Training seconds</text>
  <text x="24" y="82" class="label">Stage A</text><rect x="170" y="64" width="137" height="24" class="param"/><rect x="170" y="94" width="176" height="24" class="time"/><text x="315" y="82" class="value">1.24M</text><text x="354" y="112" class="value">1712s</text>
  <text x="24" y="142" class="label">Stage B1</text><rect x="170" y="124" width="80" height="24" class="param"/><rect x="170" y="154" width="168" height="24" class="time"/><text x="258" y="142" class="value">0.72M</text><text x="346" y="172" class="value">1638s</text>
  <text x="24" y="202" class="label">Stage B2</text><rect x="170" y="184" width="204" height="24" class="param"/><rect x="170" y="214" width="123" height="24" class="time"/><text x="382" y="202" class="value">1.85M</text><text x="301" y="232" class="value">1202s</text>
  <text x="24" y="262" class="label">Stage C proxy</text><rect x="170" y="244" width="500" height="24" class="param"/><rect x="170" y="274" width="266" height="24" class="time"/><text x="678" y="262" class="value">4.53M</text><text x="444" y="292" class="value">2592s</text>
</svg>


## 11. Experiment Timeline

<svg width="900" height="330" viewBox="0 0 900 330" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Experiment timeline diagram">
  <style>.title{font:700 18px sans-serif;fill:#111827}.box{fill:#f9fafb;stroke:#9ca3af;stroke-width:1.5;rx:8}.ok{fill:#ecfdf5;stroke:#16a34a}.warn{fill:#fff7ed;stroke:#f97316}.text{font:12px sans-serif;fill:#111827}.head{font:700 13px sans-serif;fill:#111827}.arrow{stroke:#6b7280;stroke-width:2;marker-end:url(#arrow)}</style>
  <defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#6b7280"/></marker></defs>
  <text x="24" y="30" class="title">Research Workflow</text>
  <rect x="24" y="60" width="150" height="70" class="box ok"/><text x="42" y="84" class="head">Setup</text><text x="42" y="106" class="text">Branch + config</text><text x="42" y="122" class="text">Commit ac9f370</text>
  <line x1="174" y1="95" x2="214" y2="95" class="arrow"/>
  <rect x="214" y="60" width="150" height="70" class="box ok"/><text x="232" y="84" class="head">Data</text><text x="232" y="106" class="text">Kaggle download</text><text x="232" y="122" class="text">80/20 split</text>
  <line x1="364" y1="95" x2="404" y2="95" class="arrow"/>
  <rect x="404" y="60" width="150" height="70" class="box warn"/><text x="422" y="84" class="head">Fix runtime</text><text x="422" y="106" class="text">Install deps</text><text x="422" y="122" class="text">Fix CUDA path</text>
  <line x1="554" y1="95" x2="594" y2="95" class="arrow"/>
  <rect x="594" y="60" width="150" height="70" class="box ok"/><text x="612" y="84" class="head">Stage A</text><text x="612" y="106" class="text">CNN</text><text x="612" y="122" class="text">Test .7837</text>
  <line x1="669" y1="130" x2="669" y2="166" class="arrow"/>
  <rect x="594" y="166" width="150" height="70" class="box ok"/><text x="612" y="190" class="head">Stage B1</text><text x="612" y="212" class="text">Frozen EffNet</text><text x="612" y="228" class="text">Test .7869</text>
  <line x1="594" y1="201" x2="554" y2="201" class="arrow"/>
  <rect x="404" y="166" width="150" height="70" class="box ok"/><text x="422" y="190" class="head">Stage B2</text><text x="422" y="212" class="text">Fine-tune EffNet</text><text x="422" y="228" class="text">Test .8349</text>
  <line x1="404" y1="201" x2="364" y2="201" class="arrow"/>
  <rect x="214" y="166" width="150" height="70" class="box ok"/><text x="232" y="190" class="head">Stage C proxy</text><text x="232" y="212" class="text">32-patch LSTM</text><text x="232" y="228" class="text">Test .7548</text>
  <line x1="214" y1="201" x2="174" y2="201" class="arrow"/>
  <rect x="24" y="166" width="150" height="70" class="box ok"/><text x="42" y="190" class="head">Save results</text><text x="42" y="212" class="text">results.tsv</text><text x="42" y="228" class="text">metrics + ckpts</text>
</svg>


## 12. Training Behavior Summary

| Run | Early stopping / final epoch | Best checkpoint basis | Observed behavior |
|---|---:|---|---|
| Stage A CNN | Early stopped at epoch 34 of 40 | Best validation loss around epoch 24 | Validation accuracy peaked near `0.7966`, final checkpoint evaluated at `0.7679`; strong generalization for simple model |
| Stage B1 frozen EfficientNet | Early stopped at epoch 34 of 40 | Best validation loss around epoch 24 | Validation accuracy was lower than CNN, but test accuracy still met benchmark |
| Stage B2 fine-tuned EfficientNet | Completed epoch 25 of 25 | Best validation loss around epoch 24 | Low `1e-5` learning rate improved slowly; test accuracy was the best overall despite validation accuracy trailing CNN |
| Stage C 32-patch LSTM proxy | Early stopped at epoch 54 of 60 | Best validation loss around epoch 42 | Validation accuracy peaked above `0.75`; final checkpoint met the Stage C test target by a small margin |

The training engine selects the best state by validation loss, not validation accuracy. That explains cases where the displayed peak validation accuracy during training is higher than the final validation accuracy stored in `metrics.json`.


## 13. Artifacts Produced

Successful run directories:

| Run | Directory | Key files |
|---|---|---|
| Stage A CNN | `outputs/runs/a_custom_cnn/` | `metrics.json`, `resolved_config.json`, `best_model.pt`, training curves, confusion matrices |
| Stage B1 EfficientNet frozen | `outputs/runs/b1_frozen_efficientnet/` | `metrics.json`, `resolved_config.json`, `best_model.pt`, training curves, confusion matrices |
| Stage B2 EfficientNet fine-tuned | `outputs/runs/b2_finetuned_efficientnet/` | `metrics.json`, `resolved_config.json`, `best_model.pt`, training curves, confusion matrices |
| Stage C patch LSTM proxy | `outputs/runs/c2_patch_lstm_32/` | `metrics.json`, `resolved_config.json`, `best_model.pt`, training curves, confusion matrices |

Other saved files:

- `results.tsv`: required AutoResearch run log with crashes and successful benchmark runs.
- `data/processed/prepare_report.json`: dataset validation report from `prepare.py`.
- `run.log`: last redirected training log from the Stage C proxy run.


## 14. Interpretation

The benchmark goal was achieved without adding new model families or making invasive source changes. The best model depends on the metric:

| Decision lens | Best choice | Reason |
|---|---|---|
| Highest validation accuracy | Stage A CNN | `0.767908` validation accuracy, simplest architecture, smallest parameter count |
| Highest test accuracy | Stage B2 fine-tuned EfficientNetB0 | `0.834936` test accuracy, clear margin over B2 benchmark |
| Best simplicity/performance tradeoff | Stage A CNN | Strong validation/test result with ~1.24M parameters |
| Best benchmark coverage | Full run set | Every target stage was represented and met on test accuracy |

The validation/test mismatch for B2 suggests the stratified validation split may not fully match the held-out Kaggle test distribution. Since `program.md` names validation accuracy as the primary optimization metric, Stage A is the strongest retained model by that criterion. Since the same document lists target benchmarks as test accuracy, B2 is the strongest benchmark result.


## 15. Limitations and Caveats

- Stage C is a proxy, not a true ViT. The repository did not contain a ViT implementation, and the instruction was to stick with existing models.
- B2 did not improve validation accuracy over Stage A or B1, but it produced the best test accuracy.
- The environment required a manual CUDA library path prefix. Future runs on the same VM should preserve that prefix or repair the system CUDA/PyTorch library ordering.
- The final `results.tsv` is intentionally uncommitted, per `program.md`.
- Generated dataset and outputs are ignored by git to avoid committing large artifacts.


## 16. Optional Replotting Code (Not Executed)

The following code cell is included only for future reproducibility. It has not been executed in this notebook and has no outputs. It can regenerate a standard bar chart from the static results table if needed later.


In [ ]:
# Not executed during report creation.
import pandas as pd
import matplotlib.pyplot as plt

results = pd.DataFrame([
    {"stage": "Stage A", "val_accuracy": 0.767908, "test_accuracy": 0.783654},
    {"stage": "Stage B1", "val_accuracy": 0.741165, "test_accuracy": 0.786859},
    {"stage": "Stage B2", "val_accuracy": 0.734479, "test_accuracy": 0.834936},
    {"stage": "Stage C proxy", "val_accuracy": 0.733524, "test_accuracy": 0.754808},
])

ax = results.set_index("stage")[["val_accuracy", "test_accuracy"]].plot(
    kind="bar",
    figsize=(9, 4),
    color=["#2f80ed", "#f2994a"],
)
ax.set_ylim(0.6, 0.86)
ax.set_ylabel("Accuracy")
ax.set_title("Validation vs Test Accuracy")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
